In [ ]:
# 1. Instalar librerías necesarias
!pip install librosa matplotlib tqdm

# 2. Descomprimir el archivo exacto que mencionas
import os
import zipfile

zip_path = "ESC-50-master.zip"
extract_path = "ESC-50-dataset"

if os.path.exists(zip_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("¡Archivo descomprimido con éxito!")
    except zipfile.BadZipFile:
        print(f"Error: El archivo '{zip_path}' no es un archivo ZIP válido o está corrupto. Por favor, asegúrate de que el archivo esté completo y no esté dañado, y vuelve a subirlo si es necesario.")
    except Exception as e:
        print(f"Ocurrió un error inesperado al intentar descomprimir el archivo '{zip_path}': {e}")
else:
    print(f"Error: No se encontró el archivo '{zip_path}' en la raíz de Colab. Por favor, súbelo.")

¡Archivo descomprimido con éxito!


In [ ]:
#ESTA CELDA ES PARA EXPORTAR EL CONJUNTO DE DATOS DESDE EL REPOSITORIO DE GITHUB EN CASO DE NO TENER EL CONJUNTO DE DATOS EN LA COMPUTADORA, ESTE METODO RESULTA IGUAL DE EFECTIVO

# 1. Instalar librerías necesarias
!pip install -q librosa matplotlib tqdm

# 2. Borrar el archivo corrupto (si existe)
!rm -f ESC-50-master.zip

# 3. Descargar el dataset original DIRECTAMENTE desde GitHub
print("Descargando dataset ESC-50...")
!wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O ESC-50-master.zip
print("¡Descarga completada!")

# 4. Descomprimir el archivo
import os
import zipfile

zip_path = "ESC-50-master.zip"
extract_path = "ESC-50-dataset"

if os.path.exists(zip_path):
    print("Descomprimiendo...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("¡Archivo descomprimido con éxito!")
else:
    print(f"Error: No se encontró el archivo.")

Descargando dataset ESC-50...
¡Descarga completada!
Descomprimiendo...
¡Archivo descomprimido con éxito!


In [ ]:

import pandas as pd
import numpy as np
import librosa
from tqdm import tqdm

# Rutas estándar dentro de la estructura extraída de ESC-50-master
base_dir = os.path.join(extract_path, "ESC-50-master")
csv_path = os.path.join(base_dir, "meta", "esc50.csv")
audio_dir = os.path.join(base_dir, "audio")

# Cargar el archivo de metadatos
df = pd.read_csv(csv_path)

X = []
y = []

print("Procesando audios y calculando Espectrogramas de Mel...")
# ESC-50 tiene audios a 44100Hz, los bajamos a 22050Hz para procesar más rápido sin perder calidad clave
SR = 22050

for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    file_path = os.path.join(audio_dir, row['filename'])

    # Cargar audio
    audio, sr = librosa.load(file_path, sr=SR)
    # Cambia n_mels de 64 a 128
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=128, hop_length=512)
    # Convertir a escala de decibelios (logarítmica)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    X.append(mel_spec_db)
    y.append(row['target'])

X = np.array(X)
y = np.array(y)

# Añadir una dimensión de canal (como si fueran imágenes en escala de grises para la CNN)
X = np.expand_dims(X, axis=-1)

print(f"\nDatos listos. Formato de X: {X.shape}, Formato de y: {y.shape}")

Procesando audios y calculando Espectrogramas de Mel...


100%|██████████| 2000/2000 [00:50<00:00, 39.67it/s]



Datos listos. Formato de X: (2000, 128, 216, 1), Formato de y: (2000,)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Separación usando la columna 'fold' del dataframe original
train_mask = df['fold'] < 5
val_mask = df['fold'] == 5

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]

# Clase personalizada para manejar los espectrogramas en PyTorch
class AudioDataset(Dataset):
    def __init__(self, features, labels, is_training=False):
        self.features = torch.tensor(features, dtype=torch.float32).permute(0, 3, 1, 2)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.is_training = is_training

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.features[idx] # Dimensiones: [1, n_mels, tiempo]

        if self.is_training:
            # 1. Añadir un poco de ruido blanco aleatorio
            if torch.rand(1).item() > 0.5:
                x = x + torch.randn_like(x) * 0.05

            # 2. SpecAugment: Enmascarar frecuencias (líneas horizontales)
            if torch.rand(1).item() > 0.5:
                num_lineas = int(torch.randint(1, 3, (1,)).item())
                for _ in range(num_lineas):
                    f_width = int(torch.randint(5, 15, (1,)).item())
                    f_start = int(torch.randint(0, x.shape[1] - f_width, (1,)).item())
                    x[:, f_start:f_start+f_width, :] = x.mean() # Rellena con el valor promedio

            # 3. SpecAugment: Enmascarar tiempo (líneas verticales)
            if torch.rand(1).item() > 0.5:
                num_lineas = int(torch.randint(1, 3, (1,)).item())
                for _ in range(num_lineas):
                    t_width = int(torch.randint(10, 25, (1,)).item())
                    t_start = int(torch.randint(0, x.shape[2] - t_width, (1,)).item())
                    x[:, :, t_start:t_start+t_width] = x.mean()

        return x, self.labels[idx]

In [ ]:
import torchvision.models as models
import torch.nn as nn

class ResNetAudioClassifier(nn.Module):
    def __init__(self, num_classes=50):
        super(ResNetAudioClassifier, self).__init__()

        # Cargamos una ResNet18 preentrenada con los mejores pesos disponibles
        # Usamos weights='DEFAULT' que es el estándar moderno en PyTorch
        self.resnet = models.resnet18(weights='DEFAULT')

        # CORRECCIÓN DE ENTRADA:
        # ResNet espera imágenes a color (3 canales: RGB). Nuestros espectrogramas tienen 1 canal.
        # Modificamos la primera capa convolucional para que acepte 1 solo canal conservando el resto
        original_conv = self.resnet.conv1
        self.resnet.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=original_conv.bias
        )

        # CORRECCIÓN DE SALIDA:
        # ResNet originalmente clasifica 1000 clases de ImageNet.
        # Reemplazamos la última capa lineal (fc) para que clasifique las 50 clases de ESC-50
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5), # Un dropout agresivo aquí ayuda muchísimo a evitar el overfitting
            nn.Linear(num_ftrs, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)

# Definir el dispositivo y montar el nuevo modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetAudioClassifier(num_classes=50).to(device)

print(f"¡Modelo ResNet preentrenado adaptado y montado con éxito en: {device}!")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 199MB/s]


¡Modelo ResNet preentrenado adaptado y montado con éxito en: cuda!


In [ ]:
import torch.optim as optimizer
from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.CrossEntropyLoss()

# Mantenemos el weight_decay para combatir el overfitting
optimizador = optimizer.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# --- CORRECCIÓN AQUÍ: Eliminamos verbose=True ---
scheduler = ReduceLROnPlateau(optimizador, mode='min', factor=0.5, patience=5)

epochs = 100

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizador.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizador.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # Evaluación con el set de validación (Fold 5)
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    # --- CORRECCIÓN AQUÍ: Multiplicar por 100 para obtener porcentaje real ---
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"Época [{epoch+1}/{epochs}] -> "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

Época [1/100] -> Train Loss: 3.1861 | Train Acc: 19.56% | Val Loss: 3.2163 | Val Acc: 22.25%
Época [2/100] -> Train Loss: 1.8811 | Train Acc: 46.56% | Val Loss: 2.6001 | Val Acc: 35.75%
Época [3/100] -> Train Loss: 1.4871 | Train Acc: 56.06% | Val Loss: 2.3776 | Val Acc: 40.25%
Época [4/100] -> Train Loss: 1.0584 | Train Acc: 68.38% | Val Loss: 1.7286 | Val Acc: 51.00%
Época [5/100] -> Train Loss: 0.8165 | Train Acc: 75.94% | Val Loss: 2.3074 | Val Acc: 41.50%
Época [6/100] -> Train Loss: 0.6779 | Train Acc: 79.56% | Val Loss: 1.4864 | Val Acc: 56.50%
Época [7/100] -> Train Loss: 0.5211 | Train Acc: 85.12% | Val Loss: 1.6536 | Val Acc: 57.50%
Época [8/100] -> Train Loss: 0.4398 | Train Acc: 87.06% | Val Loss: 1.6432 | Val Acc: 58.75%
Época [9/100] -> Train Loss: 0.3244 | Train Acc: 90.19% | Val Loss: 1.6975 | Val Acc: 61.25%
Época [10/100] -> Train Loss: 0.2678 | Train Acc: 91.75% | Val Loss: 1.7014 | Val Acc: 61.25%
Época [11/100] -> Train Loss: 0.3477 | Train Acc: 89.56% | Val Loss: 

In [ ]:
import torch

# Definir el nombre del archivo
model_save_path = "modelo_esc50_resnet.pth"

# Guardar los pesos (state_dict) del modelo entrenado
torch.save(model.state_dict(), model_save_path)

print(f"¡Modelo guardado con éxito en: {model_save_path}!")

¡Modelo guardado con éxito en: modelo_esc50_resnet.pth!


In [ ]:
import os
import torch
import librosa
import numpy as np
import pandas as pd
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode

# ==========================================
# 1. FORZAR LA CARGA DEL MODELO Y CLASES
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reconstruimos el modelo (Asegúrate de tener la celda de la clase ResNetAudioClassifier ejecutada)
try:
    modelo_prediccion = ResNetAudioClassifier(num_classes=50).to(device)
    weights_path = "modelo_esc50_resnet.pth"

    if os.path.exists(weights_path):
        modelo_prediccion.load_state_dict(torch.load(weights_path, map_location=device))
        modelo_prediccion.eval()
        print(" ¡Modelo cargado correctamente desde el archivo .pth!")
    else:
        print(f" Error: No se encontró el archivo '{weights_path}'. ¿Lo subiste a Colab?")
except NameError:
    print(" Error: No encontré la definición de 'ResNetAudioClassifier'. Ejecuta la celda donde definiste la arquitectura de la red antes de correr esta.")

# Mapeo de clases desde el CSV
csv_path = "ESC-50-dataset/ESC-50-master/meta/esc50.csv"
if os.path.exists(csv_path):
    df_meta = pd.read_csv(csv_path)
    clases_map = dict(zip(df_meta['target'], df_meta['category']))
else:
    clases_map = {i: f"Clase {i}" for i in range(50)}

# ==========================================
# 2. PIPELINE DE INFERENCIA (CLASIFICACIÓN)
# ==========================================
def clasificar_sonido(ruta_audio):
    SR = 22050
    audio, sr = librosa.load(ruta_audio, sr=SR)

    mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=128, hop_length=512)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    tensor_audio = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = modelo_prediccion(tensor_audio)
        probabilidades = torch.nn.functional.softmax(outputs, dim=1)[0]
        clase_detectada = torch.argmax(probabilidades).item()
        porcentaje_certeza = probabilidades[clase_detectada].item() * 100

    nombre_sonido = clases_map.get(clase_detectada, "Desconocido")
    print(f" Predicción: {nombre_sonido} (Código: {clase_detectada})")
    print(f" Certeza: {porcentaje_certeza:.2f}%")

# ==========================================
# 3. JAVASCRIPT Y CAPTURA DE MICRÓFONO
# ==========================================
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
async function recordAudio(time) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.stop();
  await sleep(100);
  const blob = new Blob(chunks, { type: 'audio/wav' });
  return await b2text(blob);
}
"""

def escuchar_y_clasificar(segundos=5):
    print(f"🎤 Dale permiso al navegador y haz un sonido por {segundos} segundos...")
    display(Javascript(RECORD_JS))

    try:
        audio_b64 = output.eval_js(f'recordAudio({segundos * 1000})')
        audio_bytes = b64decode(audio_b64.split(',')[1])

        archivo_grabado = "grabacion_usuario.wav"
        with open(archivo_grabado, 'wb') as f:
            f.write(audio_bytes)

        print(" ¡Grabación finalizada!")
        print("\n Analizando con la Red Neuronal...")
        clasificar_sonido(archivo_grabado)
    except Exception as e:
        print(f" Ocurrió un problema al grabar: {e}. Asegúrate de dar los permisos de micrófono en el navegador.")

# ==========================================
# 4. EJECUCIÓN
# ==========================================
if 'modelo_prediccion' in locals() or 'modelo_prediccion' in globals():
    escuchar_y_clasificar(segundos=5)

✅ ¡Modelo cargado correctamente desde el archivo .pth!
🎤 Dale permiso al navegador y haz un sonido por 5 segundos...


<IPython.core.display.Javascript object>

✅ ¡Grabación finalizada!

🔮 Analizando con la Red Neuronal...
🎯 Predicción: rooster (Código: 1)
📊 Certeza: 99.84%


/tmp/ipykernel_1027/2541548044.py:42: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(ruta_audio, sr=SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


## Pasos de Preprocesamiento

Los siguientes pasos de preprocesamiento fueron aplicados a los datos de audio:

1.  **Carga y Remuestreo de Audio**: Los archivos de audio se cargan utilizando `librosa.load` y se remuestrean a una frecuencia de muestreo de 22050 Hz (`SR = 22050`). Esto estandariza la entrada de audio y reduce la carga computacional.

2.  **Cálculo del Espectrograma Mel**: Para cada segmento de audio, se calcula un espectrograma Mel utilizando `librosa.feature.melspectrogram`. Los parámetros clave utilizados son:
    *   `n_mels=128`: El número de bandas Mel a generar, proporcionando una representación de frecuencia más rica.
    *   `hop_length=512`: El número de muestras entre fotogramas sucesivos, controlando la resolución temporal.

3.  **Conversión a Escala de Decibelios**: Los espectrogramas Mel de potencia se convierten a una escala de decibelios (logarítmica) utilizando `librosa.power_to_db`. Esto hace que las características sean más perceptualmente uniformes y puede ayudar a las redes neuronales a aprender de manera más efectiva.

4.  **Expansión de Dimensión**: Se añade una dimensión de canal adicional a los espectrogramas Mel utilizando `np.expand_dims(X, axis=-1)`. Esto transforma la forma de `(muestras, n_mels, fotogramas_tiempo)` a `(muestras, n_mels, fotogramas_tiempo, 1)`, haciéndola compatible con las capas `Conv2d` de PyTorch que esperan una dimensión de canal.

5.  **Aumento de Datos (durante el entrenamiento en `AudioDataset`)**:
    *   **Ruido Blanco Aleatorio**: Ocasionalmente se añade una pequeña cantidad de ruido blanco aleatorio a los espectrogramas para mejorar la robustez del modelo.
    *   **SpecAugment - Enmascaramiento de Frecuencia**: Se enmascaran bandas horizontales aleatorias (canales de frecuencia) (se establecen al valor medio) en el espectrograma. Esto ayuda al modelo a generalizar mejor al hacerlo menos dependiente de componentes de frecuencia específicos.
    *   **SpecAugment - Enmascaramiento de Tiempo**: Se enmascaran bandas verticales aleatorias (fotogramas de tiempo) en el espectrograma. Esto anima al modelo a aprender características que son robustas a las oclusiones en el dominio del tiempo.

6.  **Conversión a Tensor y Permutación**: Los arrays NumPy preprocesados se convierten en tensores de PyTorch y sus dimensiones se permutan (`permute(0, 3, 1, 2)`) para que coincidan con el formato `(batch_size, channels, height, width)` esperado por las capas convolucionales de PyTorch.